---
image: example.gif
pub-info:
    abstract: |
        vidigi isn't built for agent-based simulation, but with a little creativity it can still
        animate one - here using the Mesa package. Shows how to capture agent positions at each step of
        an ABS model so vidigi can render them like any other simulation.
execute: 
  enabled: true
---

# Using Vidigi to Visualise an Agent-Based Simulation (ABS)

While vidigi is not designed for use with agent-based simulations, with some creativity, it can be used for that purpose. 

In this example, we will be working with the Mesa package (https://mesa.readthedocs.io/latest/index.html).

The example itself is drawn from the Health Service Modelling Associates programme ([hsma.co.uk](https://www.hsma.co.uk)), with all credit going to Dr Daniel Chalk for the example, which has only had some minor modifications to support use with vidigi. 

First, let's define the model. 

It's worth noting that the key thing vidigi needs is the position of the entities at each point - so we have added the 'agent_reporters' line to the model. 

```
self.datacollector = DataCollector(
    model_reporters={"Mean Love Score": calculate_mean_love_score},
    agent_reporters={"Position": "pos", "Score": "love_score"},
    )
```

We've also added a stopping condition to the model so it will only run for 100 steps. 

In [ ]:
#| echo: false
import plotly.io as pio
pio.renderers.default = "notebook"

In [ ]:
from mesa import Agent, Model
from mesa.time import RandomActivation  # random order of agent actions
from mesa.space import MultiGrid  # multiple agents per cell
from mesa.datacollection import DataCollector

import random


# A class representing a 'HSMA' agent.  Note we're passing in the Agent class
# we imported from the mesa library.  Remember that this means our class here
# is inheriting from the 'parent' Agent class, and our class is the 'child',
# which inherits all the attributes and methods of the parent, but may have
# some of its own.
class HSMA_Agent(Agent):
    # Constructor
    def __init__(self, unique_id, model, prob_move):
        # Call the constructor from the parent Agent class, which will do all
        # the hard work of defining what an agent is - we just give it an ID
        # and a model that it will live in
        super().__init__(unique_id, model)

        # Randomly determine the agent's initial love of HSMA score
        self.love_score = random.uniform(0, 1)

        # Set the agent's default persuasiveness to 0.2
        self.persuasiveness = 0.2

        # Set the agent's probability of moving at any time step
        self.prob_move = prob_move

    # Agent movement method - this is called if it is determined the agent
    # is going to move on this time step
    def move(self):
        # Get a list of possible neighbouring cells to which to move
        # We use the get_neighborhood function, giving it the agent's current
        # position on the grid, stating we want a Moore neighbourhood (which
        # includes diagonals), and that we don't want to include the centre
        # (where the agent is currently) in the returned neighbourhood list
        possible_steps = self.model.grid.get_neighborhood(
            self.pos, moore=True, include_center=False
        )

        # Select new position at random
        new_position = random.choice(possible_steps)

        # Move the agent to the randomly selected new position
        self.model.grid.move_agent(self, new_position)

    # Method to talk to other HSMAs (if there are any around) and potentially
    # influence the opinion of other agents if there are other agents in the
    # same cell
    def talk(self):
        # Get list of agents in this cell.  We use the get_cell_list_contents
        # function of the grid object and pass it our current position
        cellmates = self.model.grid.get_cell_list_contents([self.pos])

        # Check if there are other agents here - if the list of cellmates is
        # greater than 1 then there must be more here than this agent
        if len(cellmates) > 1:
            # for each agent in the cell
            for inhabitant in cellmates:
                # Determine current persuasiveness of our agent (the one that's
                # acting here), with stronger emotions leading to a higher
                # persuasiveness
                if self.love_score < 0.1 or self.love_score > 0.9:
                    self.persuasiveness = 1
                elif self.love_score < 0.2 or self.love_score > 0.8:
                    self.persuasiveness = 0.8
                elif self.love_score < 0.3 or self.love_score > 0.7:
                    self.persuasiveness = 0.6
                elif self.love_score < 0.4 or self.love_score > 0.6:
                    self.persuasiveness = 0.4
                else:
                    self.persuasiveness = 0.2

                # Determine current love score of the other agent, and move
                # their score closer to our agent (and do this more
                # significantly the higher our agent's persuasiveness)
                other_agent_score = inhabitant.love_score

                # If the other agent has a higher score, bring them down closer
                # to our agent.  If they have a lower score, bring them up
                # closer to our agent.  If they have the same score, do nothing.
                if other_agent_score > self.love_score:
                    difference_in_scores = other_agent_score - self.love_score

                    inhabitant.love_score -= self.persuasiveness * difference_in_scores
                elif other_agent_score < self.love_score:
                    difference_in_scores = self.love_score - other_agent_score

                    inhabitant.love_score += self.persuasiveness * difference_in_scores
                else:
                    # Do nothing
                    pass

    # Step method - this defines which of the agent's actions will be taken
    # on a time step, and in which order
    def step(self):
        # Randomly decide whether the agent should move on this time step, based
        # on the agent's probability of moving
        if random.uniform(0, 1) < self.prob_move:
            self.move()

        # Regardless of whether or not the agent moved, it should begin its
        # talking behaviour for this time step
        self.talk()


# Class representing our ABS model
class Persuasion_Model(Model):
    # 2D Model initialisation constructor - initialise with N agents, and
    # specified width and height.  Also pass in the things we need to pass
    # to our agents when instantiating them.
    def __init__(self, N, width, height, prob_move, steps):
        self.running = True  # this code is required for BatchRunner
        self.num_agents = N
        self.steps = steps

        # Set up a Toroidal multi-grid (Toroidal = if the agent is in a cell
        # on the border of the grid, and moves towards the border, they'll
        # come out the other side.  Think PacMan :) The True Boolean passed in
        # switches that on.  Multi-Grid just means we can have more than one
        # agent per cell)
        self.grid = MultiGrid(width, height, True)

        # Set up a scheduler with random order of agents being activated
        # each turn.  A random activation is probably the best in most cases,
        # unless you have information that certain agents will act before
        # certain other agents
        self.schedule = RandomActivation(self)

        # Create HSMA agents up to the number specified
        for i in range(self.num_agents):
            # Create agent with ID taken from for loop - we pass in the i
            # value as the unique_id, self (the model here) as the model, and
            # then the various parameter values we specified
            a = HSMA_Agent(i, self, prob_move)

            # Add the agent to the scheduler
            self.schedule.add(a)

            # Try adding the agent to a random empty cell
            try:
                start_cell = self.grid.find_empty()
                self.grid.place_agent(a, start_cell)
            # If you can't find an empty cell, just pick any cell at random
            except:
                x = random.randrange(self.grid.width)
                y = random.randrange(self.grid.height)
                self.grid.place_agent(a, (x, y))

        # We set up a DataCollector that can collect agent-specific and
        # model-wide data.  Here, we tell it to collect data on the total number
        # of agents loving and hating HSMA, which it calculates by calling the
        # functions we name here (we write them further down).
        # We don't use any agent reporters here, but we could if we wanted to
        # track an attribute of agents over time
        self.datacollector = DataCollector(
            model_reporters={"Mean Love Score": calculate_mean_love_score},
            agent_reporters={"Position": "pos", "Score": "love_score"},
        )

    # Function to advance the model by one step (we just tell the scheduler to
    # step forward one time step)
    def step(self):
        # Tell the data collector to collect the data for this time step
        self.datacollector.collect(self)

        # Ask scheduler to step forward one time step
        self.schedule.step()

        # Stop after 100 steps
        if self.schedule.steps >= self.steps:
            self.running = False


# Function used by the data collector to calculate the mean love score across
# agents.  This will run at each time step.
def calculate_mean_love_score(model):
    list_of_love_scores = [agent.love_score for agent in model.schedule.agents]

    mean_love_score = sum(list_of_love_scores) / len(list_of_love_scores)

    return mean_love_score


Now we can start to work on the vidigi elements. First, let's do our additional imports. 

In [ ]:
import pandas as pd
import vidigi

Now we will run our model and return the agent data.

In [ ]:
model = Persuasion_Model(N=20, width=10, height=10, prob_move=0.5, steps=100)

model.run_model()

agent_results = model.datacollector.get_agent_vars_dataframe()


Let's take a look at the agent data.

In [ ]:
agent_results.head()

We can see that this is a multilevel dataframe that gives the step (which we can consider as time), agent identifier (which we can use as our entity ID), position, and - in this case - a score indicating how much participants love a training programme after interacting with each other. 

Let's reset the index of this dataframe so we have one row per step per agent. 

In [ ]:
df = agent_results.reset_index()
df

Next, we want to get the x and y coordinates into separate columns. 

It's important we don't call them "x" and "y" at this point as it will cause conflicts with some of the data preparation functions in vidigi. 

In [ ]:
df["x_coord_actual"] = df["Position"].apply(lambda x: x[0])
df["y_coord_actual"] = df["Position"].apply(lambda x: x[1])

Now we need to give each step an event type. For vidigi purposes, we'll consider these to be queues, which can be used for anything that doesn't have a resource associated with it. 

In [ ]:
df["event"] = "step"
df["event_type"] = "queue"

Our steps start at 0, but it would be better to have them starting at 1, making 0 an 'arrival' step for vidigi's purposes. 

In [ ]:
df["Step"] = df["Step"] + 1

We'll now create a dataframe that adds an arrival step for each entity. 

In [ ]:
arrivals = df[["AgentID"]].drop_duplicates().copy()
arrivals["event"] = "arrival"
arrivals["event_type"] = "arrival_departure"
arrivals["Step"] = 0


We do the same to create a departure step for each entity. 

In [ ]:
departures = df[["AgentID"]].drop_duplicates().copy()
departures["event"] = "depart"
departures["event_type"] = "arrival_departure"
departures["Step"] = max(df["Step"]+1)

We now need to join all of these together. 

In [ ]:
final_df = pd.concat([arrivals, df, departures])

final_df

Now we need to create an event position dataframe. While the event names need to match up to what we've used, everything else (x and y coordinates) is a placeholder that can really be ignored. 

In [ ]:
event_position_df = vidigi.utils.create_event_position_df([
    vidigi.utils.EventPosition(event='arrival', x=50, y=300, label="Arrival"),
    vidigi.utils.EventPosition(event='step', x=205, y=275, label="Existing"),
    vidigi.utils.EventPosition(event='depart', x=270, y=70, label="Exit")
])

Now we can pass this into our reshaping function.

In [ ]:
full_patient_df = vidigi.prep.reshape_for_animations(
    event_log=final_df,
    every_x_time_units=1,
    entity_col_name="AgentID", # pass in the custom entity column name
    time_col_name="Step", # pass in our proxy for time
    step_snapshot_max=max(final_df["AgentID"])+1, # make sure this is one higher than the number of entities
    limit_duration=max(final_df["Step"]), # final step = maximum animation duration
    debug_mode=True
    )

full_patient_df.sample(25)

Now we need to join it up to our event position dataframe using the generate_animation_df function - but we will be overwriting the x and y coordinates with the coordinates from the mesa simulation straight after.

In [ ]:
full_patient_df_plus_pos = vidigi.prep.generate_animation_df(
    full_entity_df=full_patient_df,
    event_position_df=event_position_df,
    entity_col_name="AgentID",
    time_col_name="Step",
    step_snapshot_max=max(final_df["AgentID"])+1,
    gap_between_entities=10,
    gap_between_resources=10,
    gap_between_resource_rows=30,
    gap_between_queue_rows=30,
    debug_mode=True
    )

full_patient_df_plus_pos.sort_values(['AgentID', 'snapshot_time']).head(15)

Here, we drop the 'x_final' and 'y_final' columns created by the vidigi function. 

We then rename our x and y coordinate columns we created previously to 'x_final' and 'y_final' so the animation function will use these to determine the position of every entity at every point. 

Alternatively, you could skip creating the x and y columns earlier, and create them at this point from the step column (being careful to ensure they are named 'x_final' and 'y_final'). 

In [ ]:
full_patient_df_plus_pos = (
    full_patient_df_plus_pos
    .drop(columns=["x_final", "y_final"])
    .rename(columns={"x_coord_actual":"x_final", "y_coord_actual": "y_final"})
    )


Now we can animate. 

In [ ]:
fig = vidigi.animation.generate_animation(
        full_entity_df_plus_pos=full_patient_df_plus_pos.sort_values(['AgentID', 'snapshot_time']),
        event_position_df= event_position_df,
        scenario=None,
        entity_col_name="AgentID",
        time_col_name="Step",
        debug_mode=True,
        setup_mode=True,
        include_play_button=True,
        entity_icon_size=16,
        plotly_height=600,
        frame_duration=800,
        frame_transition_duration=200,
        plotly_width=1000,
        override_x_max=12,
        override_y_max=12,
        time_display_units="dhm",
        display_stage_labels=False,
        # We provide custom hover text so that the score is displayed
        hover_text_entity="Entity: %{customdata[0]}<br>Score: %{customdata[1]}",
        custom_hover_data=["AgentID","Score"],
    )

# We can't set the lower x and y axis range in the vidigi function, so we override it afterwards
fig.update_xaxes(range=[-2, 12])
fig.update_yaxes(range=[-2, 12])

fig

This is a start, but it doesn't really tell us anything at this stage. 

Let's bring in some custom emojis and assign them to the output of generate_animation_df() before we pass that to generate_animation(). 

In [ ]:
def agent_portrayal(row):
    # Set colour of agent according to their love score
    if row["Score"] < 0.2:
        return "😡"
    elif row["Score"] < 0.4:
        return "😒"
    elif row["Score"] < 0.6:
        return "😐"
    elif row["Score"] < 0.8:
        return "😁"
    else:
        return "❤️"

In [ ]:
full_patient_df_plus_pos = full_patient_df_plus_pos.assign(
            icon=full_patient_df_plus_pos.apply(agent_portrayal, axis=1)
            )

In [ ]:
fig = vidigi.animation.generate_animation(
        full_entity_df_plus_pos=full_patient_df_plus_pos.sort_values(['AgentID', 'snapshot_time']),
        event_position_df= event_position_df,
        scenario=None,
        entity_col_name="AgentID",
        time_col_name="Step",
        debug_mode=True,
        setup_mode=False,
        include_play_button=True,
        entity_icon_size=16,
        resource_icon_size=20,
        gap_between_resource_rows=30,
        plotly_height=600,
        frame_duration=600,
        frame_transition_duration=200,
        plotly_width=1000,
        override_x_max=12,
        override_y_max=12,
        time_display_units="dhm",
        display_stage_labels=False,
        # We provide custom hover text so that the score is displayed
        hover_text_entity="Entity: %{customdata[0]}<br>Score: %{customdata[1]}",
        custom_hover_data=["AgentID","Score"],
    )

fig.update_xaxes(range=[-2, 12])
fig.update_yaxes(range=[-2, 12])

This is much better - but while they intuitively make sense, it's not very easy to get a quick sense of the general distribution of scores. 

Instead, we'll just use coloured dots. 

In [ ]:
def agent_portrayal(row):
    # Set colour of agent according to their love score
    if row["Score"] < 0.2:
        return "⚪"
    elif row["Score"] < 0.4:
        return "🟤"
    elif row["Score"] < 0.6:
        return "🔵"
    elif row["Score"] < 0.8:
        return "🟠"
    else:
        return "🔴"

In [ ]:
full_patient_df_plus_pos = full_patient_df_plus_pos.assign(
            icon=full_patient_df_plus_pos.apply(agent_portrayal, axis=1)
            )

In [ ]:
fig = vidigi.animation.generate_animation(
        full_entity_df_plus_pos=full_patient_df_plus_pos.sort_values(['AgentID', 'snapshot_time']),
        event_position_df= event_position_df,
        scenario=None,
        entity_col_name="AgentID",
        time_col_name="Step",
        debug_mode=True,
        setup_mode=False,
        include_play_button=True,
        entity_icon_size=16,
        resource_icon_size=20,
        gap_between_resource_rows=30,
        plotly_height=600,
        frame_duration=600,
        frame_transition_duration=200,
        plotly_width=1000,
        override_x_max=12,
        override_y_max=12,
        time_display_units="dhm",
        display_stage_labels=False,
        # We provide custom hover text so that the score is displayed
        hover_text_entity="Entity: %{customdata[0]}<br>Score: %{customdata[1]}",
        custom_hover_data=["AgentID","Score"],
    )

fig.update_xaxes(range=[-2, 12])
fig.update_yaxes(range=[-2, 12])

This is much better - we can now get a quick intuitive sense of the distribution of scores. 

Let's now explore adding a synchronised line plot that shows the average score over time. This a bit complex, but the general pattern holds true for any additional plots you wish to add, and it's simpler in this case than in an animation with resources as we only have a single animated layer to worry about to begin with.

First let's create a dataframe containing the data we want to plot as a line.

In [ ]:
mean_score_per_step = pd.DataFrame(agent_results.reset_index().groupby("Step")["Score"].mean()).reset_index()
mean_score_per_step

Now let's do some additional plotting imports.

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

Here we split our figure into two subplots. Plotly doesn't really like doing this after a plot has already been created, which is why it's a little complex. 

In [ ]:
# Set up the desired subplot layout
ROWS = 2

sp = make_subplots(
    rows=ROWS,
    cols=1,
    row_heights=[0.75, 0.25],
    vertical_spacing=0.05,
    subplot_titles=(
        "", # Original Animation
        "Station Tank Fuel Level", # Fuel Tank Level
        )
    )

# Overwrite the domain of our original x and y axis with domain from the new axis
fig.layout['xaxis']['domain'] = sp.layout['xaxis']['domain']
fig.layout['yaxis']['domain'] = sp.layout['yaxis']['domain']

for i in range(2, ROWS+1):

    # Add in the attributes for the secondary axis from our subplot
    fig.layout[f'xaxis{i}'] = sp.layout[f'xaxis{i}']
    fig.layout[f'yaxis{i}'] = sp.layout[f'yaxis{i}']

fig._grid_ref = sp._grid_ref

Now we want to set up our additional trace - our animated line plot. 

First, we need to set it up on the first frame only. 

In [ ]:
# Keep our figure data as just the initial trace.
fig.data = (fig.data[0],)

# LINE PLOT ON SECONDARY AXIS (animated line plot in subplot)
# Initialize with a single point and assign it to subplot axes (x2/y2)

mean_score_per_step = mean_score_per_step.sort_values("Step")
steps = mean_score_per_step["Step"].values
scores = mean_score_per_step["Score"].values

# Get unique time points
time_points = final_df["Step"].unique()

# Initial frame (first time point)
initial_time = time_points[0]
initial_df = final_df[final_df["Step"] == initial_time]

# Create the initial line trace

fig = fig.add_trace(go.Scatter(
    x=[steps[0]],
    y=[scores[0]],
    showlegend=False,
    mode="lines+markers",
    line=dict(width=2),
    marker=dict(size=4),
    # We place it in our new subplot using the following line
), row=2, col=1)

fig = fig.update_xaxes(range=[0, 100], row=2, col=1, autorange=False)
fig = fig.update_yaxes(range=[0, 1], row=2, col=1, autorange=False)


Now we need to add it to each subsequent frame. 

In [ ]:
# # Now ensure we tell it which traces we are animating
# # (as per https://chart-studio.plotly.com/~empet/15243/animating-traces-in-subplotsbr/#/)
for i, frame in enumerate(fig.frames):
    original_data = frame.data

    # The new data you want to add for this specific frame
    new_data = (
        go.Scatter(
            x=steps[: i + 1],
            y=scores[: i + 1],
            mode="lines+markers",
            line=dict(width=2),
            marker=dict(size=4),
            showlegend=False
        ) ,  # This needs to be a tuple even if we're only adding a single additional trace, hence the comma
    )

    frame.data = original_data + new_data

Finally, we can view our new two-in-one plot. 

In [ ]:
fig

# Sickness example

Finally, let's just imagine what this might look like to visualise healthy and sick populations - a common use for ABS. We'll keep our simulation the same - it's just for demo purposes.

However, we will increase the number of agents and the size of the grid, as well as the duration. 

In [ ]:
model = Persuasion_Model(N=1000, width=50, height=50, prob_move=0.2, steps=1000)

model.run_model()

agent_results = model.datacollector.get_agent_vars_dataframe()

df = agent_results.reset_index()

df["x_coord_actual"] = df["Position"].apply(lambda x: x[0])
df["y_coord_actual"] = df["Position"].apply(lambda x: x[1])

df["event"] = "step"
df["event_type"] = "queue"

df["Step"] = df["Step"] + 1

arrivals = df[["AgentID"]].drop_duplicates().copy()
arrivals["event"] = "arrival"
arrivals["event_type"] = "arrival_departure"
arrivals["Step"] = 0

departures = df[["AgentID"]].drop_duplicates().copy()
departures["event"] = "depart"
departures["event_type"] = "arrival_departure"
departures["Step"] = max(df["Step"]+1)

final_df = pd.concat([arrivals, df, departures])

full_patient_df = vidigi.prep.reshape_for_animations(
    event_log=final_df,
    every_x_time_units=1,
    entity_col_name="AgentID", # pass in the custom entity column name
    time_col_name="Step", # pass in our proxy for time
    step_snapshot_max=max(final_df["AgentID"])+1, # make sure this is one higher than the number of entities
    limit_duration=max(final_df["Step"]), # final step = maximum animation duration
    debug_mode=True
    )

full_patient_df_plus_pos = vidigi.prep.generate_animation_df(
    full_entity_df=full_patient_df,
    event_position_df=event_position_df,
    entity_col_name="AgentID",
    time_col_name="Step",
    step_snapshot_max=max(final_df["AgentID"])+1,
    gap_between_entities=10,
    gap_between_resources=10,
    gap_between_resource_rows=30,
    gap_between_queue_rows=30,
    debug_mode=True
    )

def agent_portrayal(row):
    # Set colour of agent
    if row["Score"] < 0.2:
        return "🤢"
    else:
        return "😊"

full_patient_df_plus_pos = full_patient_df_plus_pos.assign(
            icon=full_patient_df_plus_pos.apply(agent_portrayal, axis=1)
            )

full_patient_df_plus_pos = (
    full_patient_df_plus_pos
    .drop(columns=["x_final", "y_final"])
    .rename(columns={"x_coord_actual":"x_final", "y_coord_actual": "y_final"})
    )


In [ ]:
full_patient_df_plus_pos

In [ ]:
fig = vidigi.animation.generate_animation(
        full_entity_df_plus_pos=full_patient_df_plus_pos.sort_values(['AgentID', 'snapshot_time']),
        event_position_df= event_position_df,
        scenario=None,
        entity_col_name="AgentID",
        time_col_name="Step",
        debug_mode=True,
        setup_mode=False,
        include_play_button=True,
        entity_icon_size=8,
        plotly_height=800,
        frame_duration=200,
        frame_transition_duration=200,
        plotly_width=800,
        override_x_max=12,
        override_y_max=12,
        simulation_time_unit="days",
        display_stage_labels=False,
        # We provide custom hover text so that the score is displayed
        hover_text_entity="Entity: %{customdata[0]}<br>Score: %{customdata[1]}",
        custom_hover_data=["AgentID","Score"],
    )

fig.update_xaxes(range=[-2, 52])
fig.update_yaxes(range=[-2, 52])